## Unlabeled Data

This Notebook uses the MassSpecGym datasets to create test data for the unlabeled input. This data includes:
1. Unlabeled candidates (for precomputation)
2. Spectra (for online retrieval)

both will be constructed with [MassSpecGym_retrieval_candidates_formula.json](https://huggingface.co/datasets/roman-bushuiev/MassSpecGym/blob/main/data/molecules/MassSpecGym_retrieval_candidates_formula.json) and the official [MassSpecGym dataset](https://huggingface.co/datasets/roman-bushuiev/MassSpecGym)

In [ ]:
import json
import pandas as pd
from jestr.data.datasets import MassSpecDataset, ExpandedRetrievalDataset
#TODO what is molfeaturizer used for why is it directly called in test.py

#### 1. Load dataset
- uses the original workflow to extend MassSpecGym dataset with candidates
- with that build than the sepparate datasets

In [ ]:
# Load ExpandedRetrievalDataset
# pass spectra_view default 
spectra_view = "SpecBinnerLog"
erd = ExpandedRetrievalDataset(spectra_view=spectra_view)
print(f"ExpandedRetrievalDataset instantiated. spectra metadata rows: {len(erd.instance.metadata)}")


In [ ]:
print(f"length of spectra: {len(erd.spectra)}")
print(f"length of dataset: {len(erd)}")

#### 3. Split dataset

In [ ]:
# Creates two datasets from the ExpandedRetrievalDataset instance,
#  one for unlabeled candidates and one for spectra information
# only take data entries with test labels

# 1. Unlabeled candidates (for precomputation)
# Select all candidates in the found Dataset by keeping their order
# export a list of list of candidates to a json file
# collect candidates into a list of lists (one list of candidates per spectrum)
from collections import defaultdict
import json

def build_list_of_lists_of_candidates(erd):
    grouped = defaultdict(list)

    # spec_cand contains tuples: (spec_idx, cand_smiles, label)
    for spec_idx, cand_smiles, label in erd.spec_cand:
        grouped[spec_idx].append(cand_smiles)

    # reorder by spectrum index (0..len(erd.spectra)-1)
    list_of_lists = [grouped[i] for i in range(len(erd.spectra))]
    return list_of_lists

list_of_lists_of_candidates = build_list_of_lists_of_candidates(erd)

with open("data/sample/list_of_lists_of_candidates.json", "w") as f:
    json.dump(list_of_lists_of_candidates, f, indent=2)

print("Saved", len(list_of_lists_of_candidates), "candidate lists.")

# 2. Spectra (for online retrieval)
# Select all spectra and their information in the found Dataset
# exclude all candidates and target molecule information
# include only the following spectra information:
#identifier mzs intensities precursor_formula parent_mass precursor_mz adduct instrument_type collision_energy simulation_challenge
from matchms import Spectrum
from matchms.exporting import save_as_json
import numpy as np

spectra_out = []

for i, spec in enumerate(erd.spectra):
    meta = erd.metadata.iloc[i]

    # Build a new Spectrum object with mz/intensity arrays
    new_spec = Spectrum(
        mz=np.array(spec.peaks.mz, dtype=float),
        intensities=np.array(spec.peaks.intensities, dtype=float),
        metadata={
            "identifier": meta["identifier"],
            "precursor_formula": meta.get("precursor_formula", ""),
            "parent_mass": meta.get("parent_mass", ""),
            "precursor_mz": meta.get("precursor_mz", ""),
            "adduct": meta.get("adduct", ""),
            "instrument_type": meta.get("instrument_type", ""),
            "collision_energy": meta.get("collision_energy", ""),
            "simulation_challenge": meta.get("simulation_challenge", ""),
        }
    )

    spectra_out.append(new_spec)

# Save as matchms-compatible JSON
output_path = "data/sample/expanded_retrieval_dataset.json"
save_as_json(spectra_out, output_path)

print(f"Saved {len(spectra_out)} spectra to {output_path}")

In [ ]:
erd.metadata["precursor_mz"].isna().sum()
